# WC_MERCURY_BADGEEVENT_F ETL - ODI to Databricks Migration

**Original ODI Package:** WC_MERCURY_BADGEEVENT_F Load (Fact Table)

**Source Schema:** `workspace.PRXBI_TS`

**Target Schema:** `workspace.PRXBI_DW`

**Detection Strategy:** NONE (Upsert based on INTEGRATION_ID)

**Load Type:** Incremental Fact Load with Dimension Lookups

---

## Step 1: Define Widgets/Parameters

In [ ]:
%sql
-- Create widgets for runtime parameters
CREATE WIDGET TEXT ETL_JOB_TYPE DEFAULT 'EOD';
CREATE WIDGET TEXT DATASOURCE_NUM_ID DEFAULT '380';
CREATE WIDGET TEXT ETL_PROC_WID DEFAULT '1';

---

## Step 2: Get ETL Control Parameters

In [ ]:
%sql
-- Get last extract time
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT etl_last_extract_time 
FROM workspace.PRXBI_DW.wc_etl_parameters 
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [ ]:
%sql
-- Get current extract time
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT etl_current_extract_time 
FROM workspace.PRXBI_DW.wc_etl_parameters 
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [ ]:
%sql
-- Get ROW_WID for ETL parameters
CREATE OR REPLACE TEMPORARY VIEW v_etl_row_wid AS
SELECT ROW_WID 
FROM workspace.PRXBI_DW.wc_etl_parameters 
WHERE ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [ ]:
%sql
-- Display ETL parameters for verification
SELECT 
    'Last Extract Time' AS parameter,
    CAST(etl_last_extract_time AS STRING) AS value
FROM v_etl_last_extract_time
UNION ALL
SELECT 
    'Current Extract Time' AS parameter,
    CAST(etl_current_extract_time AS STRING) AS value
FROM v_etl_current_extract_time
UNION ALL
SELECT 
    'ETL ROW_WID' AS parameter,
    CAST(ROW_WID AS STRING) AS value
FROM v_etl_row_wid;

---

## Step 3: Create Staging Table (C$)

In [ ]:
%sql
-- Drop staging table if exists
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_badgeevent_stg;

In [ ]:
%sql
-- Create staging table for badge events
CREATE TABLE workspace.PRXBI_DW.c_badgeevent_stg (
    EVENTEDITIONGBSCODE STRING,
    EVENTTYPE STRING,
    BADGEID STRING,
    SOURCE STRING,
    PRODUCTCODE STRING,
    CUSTOMERTYPE STRING,
    EVENTDATE STRING,
    CREATEDDATE STRING
)
USING DELTA;

---

## Step 4: Extract Incremental Data with Deduplication

In [ ]:
%sql
-- Extract incremental badge event data with deduplication
-- CRITICAL: Preserves original ODI late-arriving data logic exactly
--
-- Business Rule (load-time precedence):
--   1. FIRST priority: Select records with MAX(INT_INSERT_DATE) per composite key
--   2. SECOND priority: Among those, pick the one with latest EVENTDATE (tiebreaker)
--
-- This handles late-arriving data correctly: a record loaded later (higher INT_INSERT_DATE)
-- takes precedence over earlier loads, even if it has an older EVENTDATE.

INSERT INTO workspace.PRXBI_DW.c_badgeevent_stg
SELECT
    EVENTEDITIONGBSCODE,
    EVENTTYPE,
    BADGEID,
    SOURCE,
    PRODUCTCODE,
    CUSTOMERTYPE,
    EVENTDATE,
    CREATEDDATE
FROM (
    SELECT
        TS.EVENTEDITIONGBSCODE,
        TS.EVENTTYPE,
        TS.BADGEID,
        TS.SOURCE,
        TS.PRODUCTCODE,
        TS.CUSTOMERTYPE,
        TS.EVENTDATE,
        TS.CREATEDDATE,
        -- ROW_NUMBER with EVENTDATE DESC is applied AFTER filtering to MAX(INT_INSERT_DATE)
        ROW_NUMBER() OVER (
            PARTITION BY
                TS.EVENTEDITIONGBSCODE,
                TS.EVENTTYPE,
                TS.BADGEID,
                TS.SOURCE,
                TS.PRODUCTCODE,
                TS.CUSTOMERTYPE
            ORDER BY TS.EVENTDATE DESC
        ) AS RN
    FROM workspace.PRXBI_TS.wc_mercury_badgeevent_ts TS
    -- Step 1: INNER JOIN to get only records with MAX(INT_INSERT_DATE) per composite key
    INNER JOIN (
        SELECT
            EVENTEDITIONGBSCODE,
            EVENTTYPE,
            BADGEID,
            SOURCE,
            PRODUCTCODE,
            CUSTOMERTYPE,
            MAX(INT_INSERT_DATE) AS MAX_INT_INSERT_DATE
        FROM workspace.PRXBI_TS.wc_mercury_badgeevent_ts
        WHERE INT_INSERT_DATE > (SELECT etl_last_extract_time FROM v_etl_last_extract_time)
          AND INT_INSERT_DATE <= (SELECT etl_current_extract_time FROM v_etl_current_extract_time)
        GROUP BY
            EVENTEDITIONGBSCODE,
            EVENTTYPE,
            BADGEID,
            SOURCE,
            PRODUCTCODE,
            CUSTOMERTYPE
    ) AGG ON TS.EVENTEDITIONGBSCODE = AGG.EVENTEDITIONGBSCODE
         AND TS.EVENTTYPE = AGG.EVENTTYPE
         AND TS.BADGEID = AGG.BADGEID
         AND TS.SOURCE = AGG.SOURCE
         AND TS.PRODUCTCODE = AGG.PRODUCTCODE
         AND TS.CUSTOMERTYPE = AGG.CUSTOMERTYPE
         AND TS.INT_INSERT_DATE = AGG.MAX_INT_INSERT_DATE
    WHERE TS.INT_INSERT_DATE > (SELECT etl_last_extract_time FROM v_etl_last_extract_time)
      AND TS.INT_INSERT_DATE <= (SELECT etl_current_extract_time FROM v_etl_current_extract_time)
)
WHERE RN = 1;

In [ ]:
%sql
-- Validation: Show staging record count
SELECT COUNT(*) AS staging_count FROM workspace.PRXBI_DW.c_badgeevent_stg;

---

## Step 5: Create Integration/Flow Table (I$)

In [ ]:
%sql
-- Drop integration table if exists
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_badgeevent_flow;

In [ ]:
%sql
-- Create integration/flow table for fact loading
CREATE TABLE workspace.PRXBI_DW.i_badgeevent_flow (
    ROW_WID DECIMAL(20,0),
    INTEGRATION_ID STRING,
    ETL_PROC_WID DECIMAL(20,0),
    EVENTEDITIONGBSCODE STRING,
    EVENTTYPE STRING,
    BADGEID STRING,
    SOURCE STRING,
    PRODUCTCODE STRING,
    CUSTOMERTYPE STRING,
    EVENTDATE STRING,
    CREATEDDATE STRING,
    BADGE_WID DECIMAL(20,0),
    EVENT_EDITION_WID DECIMAL(20,0),
    OBU_WID DECIMAL(20,0),
    PRODUCT_WID DECIMAL(20,0),
    W_UPDATE_DT TIMESTAMP,
    W_INSERT_DT TIMESTAMP,
    EVENT_WID DECIMAL(20,0),
    IND_UPDATE STRING
)
USING DELTA;

---

## Step 6: Load Integration Table with Dimension Lookups

In [ ]:
%sql
-- Insert into flow table with dimension lookups
-- Detection Strategy: NONE - all records are processed
-- Multiple LEFT OUTER JOINs to dimension tables for WID lookups

INSERT INTO workspace.PRXBI_DW.i_badgeevent_flow
SELECT 
    NULL AS ROW_WID,
    -- Composite INTEGRATION_ID: EVENTEDITIONGBSCODE~EVENTTYPE~BADGEID~SOURCE~PRODUCTCODE~CUSTOMERTYPE
    CONCAT_WS('~', 
        STG.EVENTEDITIONGBSCODE,
        STG.EVENTTYPE,
        STG.BADGEID,
        STG.SOURCE,
        STG.PRODUCTCODE,
        STG.CUSTOMERTYPE
    ) AS INTEGRATION_ID,
    NULL AS ETL_PROC_WID,
    STG.EVENTEDITIONGBSCODE,
    STG.EVENTTYPE,
    STG.BADGEID,
    STG.SOURCE,
    STG.PRODUCTCODE,
    STG.CUSTOMERTYPE,
    STG.EVENTDATE,
    STG.CREATEDDATE,
    -- Dimension Lookups with NVL -> COALESCE
    COALESCE(BADGE.ROW_WID, 0) AS BADGE_WID,
    COALESCE(EVENT_ED.ROW_WID, 0) AS EVENT_EDITION_WID,
    COALESCE(EVENT_ED.OBU_WID, 0) AS OBU_WID,
    COALESCE(PRODUCT.ROW_WID, 0) AS PRODUCT_WID,
    CURRENT_TIMESTAMP() AS W_UPDATE_DT,
    CURRENT_TIMESTAMP() AS W_INSERT_DT,
    COALESCE(EVENT_D.ROW_WID, 0) AS EVENT_WID,
    'I' AS IND_UPDATE
FROM workspace.PRXBI_DW.c_badgeevent_stg STG

-- Lookup 1: Event Edition Dimension
-- Join condition: EVENTEDITIONGBSCODE = RPAD(EVENT_ALPHA_CODE, 5, '-') || EVENT_EDITION_CODE
LEFT OUTER JOIN (
    SELECT 
        ROW_WID,
        EVENT_EDITION_CODE,
        EVENT_ALPHA_CODE,
        OBU_WID,
        EVENT_INTEGRATION_ID,
        CONCAT(RPAD(EVENT_ALPHA_CODE, 5, '-'), EVENT_EDITION_CODE) AS EVENT_ED_KEY
    FROM workspace.PRXBI_DW.wc_event_ed_d
) EVENT_ED ON STG.EVENTEDITIONGBSCODE = EVENT_ED.EVENT_ED_KEY

-- Lookup 2: Badge Details Dimension
LEFT OUTER JOIN workspace.PRXBI_DW.wc_badge_details_d BADGE 
    ON STG.BADGEID = BADGE.BADGE_ID

-- Lookup 3: Product Dimension (aggregated by SKU)
LEFT OUTER JOIN (
    SELECT 
        MAX(ROW_WID) AS ROW_WID,
        SKU
    FROM workspace.PRXBI_DW.wc_badge_product_d
    GROUP BY SKU
) PRODUCT ON STG.PRODUCTCODE = PRODUCT.SKU

-- Lookup 4: Event Dimension (via Event Edition)
LEFT OUTER JOIN workspace.PRXBI_DW.wc_event_d EVENT_D 
    ON EVENT_ED.EVENT_INTEGRATION_ID = EVENT_D.INTEGRATION_ID

WHERE (1=1);

In [ ]:
%sql
-- Validation: Show records in flow table
SELECT COUNT(*) AS flow_record_count FROM workspace.PRXBI_DW.i_badgeevent_flow;

---

## Step 7: Data Quality Check - Duplicate Detection

In [ ]:
%sql
-- Check for duplicate INTEGRATION_IDs (PK violations)
-- This replaces ODI's E$ error table and SNP_CHECK_TAB logic
CREATE OR REPLACE TEMPORARY VIEW v_duplicate_check AS
SELECT 
    INTEGRATION_ID,
    COUNT(*) AS duplicate_count
FROM workspace.PRXBI_DW.i_badgeevent_flow
GROUP BY INTEGRATION_ID
HAVING COUNT(*) > 1;

In [ ]:
%sql
-- Display duplicate records if any
SELECT 
    'PK_BADGE_EVENT' AS constraint_name,
    'Primary key is not unique' AS error_message,
    COUNT(*) AS error_count
FROM v_duplicate_check
WHERE duplicate_count > 0;

In [ ]:
%sql
-- Remove duplicates from flow table (keep first occurrence)
-- This replaces ODI's error table deletion logic
DELETE FROM workspace.PRXBI_DW.i_badgeevent_flow
WHERE INTEGRATION_ID IN (
    SELECT INTEGRATION_ID FROM v_duplicate_check
)
AND ROW_WID IS NULL  -- Only delete duplicates, not already processed records
AND INTEGRATION_ID || EVENTDATE NOT IN (
    -- Keep the record with latest EVENTDATE for each duplicate
    SELECT INTEGRATION_ID || MAX(EVENTDATE)
    FROM workspace.PRXBI_DW.i_badgeevent_flow
    WHERE INTEGRATION_ID IN (SELECT INTEGRATION_ID FROM v_duplicate_check)
    GROUP BY INTEGRATION_ID
);

---

## Step 8: Perform MERGE Operation (Upsert to Fact Table)

In [ ]:
%sql
-- MERGE into fact table
-- Detection Strategy: NONE - updates existing, inserts new based on INTEGRATION_ID
MERGE INTO workspace.PRXBI_DW.wc_mercury_badgeevent_f AS T
USING workspace.PRXBI_DW.i_badgeevent_flow AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID

-- Update existing records
WHEN MATCHED THEN UPDATE SET
    T.EVENTEDITIONGBSCODE = S.EVENTEDITIONGBSCODE,
    T.EVENTTYPE = S.EVENTTYPE,
    T.BADGEID = S.BADGEID,
    T.SOURCE = S.SOURCE,
    T.PRODUCTCODE = S.PRODUCTCODE,
    T.CUSTOMERTYPE = S.CUSTOMERTYPE,
    T.EVENTDATE = S.EVENTDATE,
    T.CREATEDDATE = S.CREATEDDATE,
    T.BADGE_WID = S.BADGE_WID,
    T.EVENT_EDITION_WID = S.EVENT_EDITION_WID,
    T.OBU_WID = S.OBU_WID,
    T.PRODUCT_WID = S.PRODUCT_WID,
    T.EVENT_WID = S.EVENT_WID,
    T.ETL_PROC_WID = ${ETL_PROC_WID},
    T.W_UPDATE_DT = CURRENT_TIMESTAMP()

-- Insert new records
WHEN NOT MATCHED THEN INSERT (
    INTEGRATION_ID,
    EVENTEDITIONGBSCODE,
    EVENTTYPE,
    BADGEID,
    SOURCE,
    PRODUCTCODE,
    CUSTOMERTYPE,
    EVENTDATE,
    CREATEDDATE,
    BADGE_WID,
    EVENT_EDITION_WID,
    OBU_WID,
    PRODUCT_WID,
    EVENT_WID,
    ETL_PROC_WID,
    W_UPDATE_DT,
    W_INSERT_DT
) VALUES (
    S.INTEGRATION_ID,
    S.EVENTEDITIONGBSCODE,
    S.EVENTTYPE,
    S.BADGEID,
    S.SOURCE,
    S.PRODUCTCODE,
    S.CUSTOMERTYPE,
    S.EVENTDATE,
    S.CREATEDDATE,
    S.BADGE_WID,
    S.EVENT_EDITION_WID,
    S.OBU_WID,
    S.PRODUCT_WID,
    S.EVENT_WID,
    ${ETL_PROC_WID},
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP()
);

---

## Step 9: Optimize & Cleanup

In [ ]:
%sql
-- Optimize fact table (replaces Oracle dbms_stats)
OPTIMIZE workspace.PRXBI_DW.wc_mercury_badgeevent_f
ZORDER BY (INTEGRATION_ID);

In [ ]:
%sql
-- Drop temporary tables
DROP TABLE IF EXISTS workspace.PRXBI_DW.i_badgeevent_flow;
DROP TABLE IF EXISTS workspace.PRXBI_DW.c_badgeevent_stg;

---

## Step 10: Validation & Summary

In [ ]:
%sql
-- Final validation - show summary statistics
SELECT 
    'WC_MERCURY_BADGEEVENT_F' AS table_name,
    COUNT(*) AS total_records,
    COUNT(DISTINCT INTEGRATION_ID) AS unique_events,
    COUNT(DISTINCT BADGEID) AS unique_badges,
    COUNT(DISTINCT EVENTEDITIONGBSCODE) AS unique_event_editions,
    MAX(W_UPDATE_DT) AS last_update_time,
    'ETL Completed Successfully' AS status
FROM workspace.PRXBI_DW.wc_mercury_badgeevent_f;

In [ ]:
%sql
-- Show dimension lookup statistics
SELECT 
    'Records with BADGE_WID = 0 (no match)' AS metric,
    COUNT(*) AS count
FROM workspace.PRXBI_DW.wc_mercury_badgeevent_f
WHERE BADGE_WID = 0
UNION ALL
SELECT 
    'Records with EVENT_EDITION_WID = 0 (no match)' AS metric,
    COUNT(*) AS count
FROM workspace.PRXBI_DW.wc_mercury_badgeevent_f
WHERE EVENT_EDITION_WID = 0
UNION ALL
SELECT 
    'Records with PRODUCT_WID = 0 (no match)' AS metric,
    COUNT(*) AS count
FROM workspace.PRXBI_DW.wc_mercury_badgeevent_f
WHERE PRODUCT_WID = 0;

In [ ]:
%sql
-- Show sample of recently loaded records
SELECT 
    INTEGRATION_ID,
    EVENTEDITIONGBSCODE,
    EVENTTYPE,
    BADGEID,
    BADGE_WID,
    EVENT_EDITION_WID,
    PRODUCT_WID,
    W_UPDATE_DT
FROM workspace.PRXBI_DW.wc_mercury_badgeevent_f
ORDER BY W_UPDATE_DT DESC
LIMIT 10;

---

## Conversion Notes

### Key Changes from ODI to Databricks:

1. **Detection Strategy NONE**: Original ODI used no change detection - all records are processed. Converted to MERGE with simple INTEGRATION_ID matching.

2. **Composite INTEGRATION_ID**: Created using `CONCAT_WS('~', ...)` instead of Oracle's `||'~'||` concatenation.

3. **Dimension Lookups**: Multiple LEFT OUTER JOINs converted:
   - `WC_EVENT_ED_D`: Join on computed key `RPAD(EVENT_ALPHA_CODE, 5, '-') || EVENT_EDITION_CODE`
   - `WC_BADGE_DETAILS_D`: Join on BADGE_ID
   - `WC_BADGE_PRODUCT_D`: Aggregated by SKU with MAX(ROW_WID)
   - `WC_EVENT_D`: Join via EVENT_INTEGRATION_ID from Event Edition

4. **NVL -> COALESCE**: `NVL(expr, 0)` converted to `COALESCE(expr, 0)`

5. **Error Handling Simplified**: 
   - Original ODI had E$ error table and SNP_CHECK_TAB for tracking PK violations
   - Converted to simple duplicate check view and deletion logic
   - Consider implementing Delta Lake constraints for production

6. **Sequences Removed**: `WC_MERCURY_BADGEEVENT_F_SEQ.NEXTVAL` removed. Use identity columns if needed.

7. **SYSTIMESTAMP -> CURRENT_TIMESTAMP()**: Standard Spark SQL function.

### Target Table Structure (if needed):
```sql
CREATE TABLE workspace.PRXBI_DW.wc_mercury_badgeevent_f (
    ROW_WID BIGINT GENERATED ALWAYS AS IDENTITY,
    INTEGRATION_ID STRING,
    ETL_PROC_WID BIGINT,
    EVENTEDITIONGBSCODE STRING,
    EVENTTYPE STRING,
    BADGEID STRING,
    SOURCE STRING,
    PRODUCTCODE STRING,
    CUSTOMERTYPE STRING,
    EVENTDATE STRING,
    CREATEDDATE STRING,
    BADGE_WID BIGINT,
    EVENT_EDITION_WID BIGINT,
    OBU_WID BIGINT,
    PRODUCT_WID BIGINT,
    EVENT_WID BIGINT,
    W_UPDATE_DT TIMESTAMP,
    W_INSERT_DT TIMESTAMP
)
USING DELTA;
```

### Dimension Tables Required:
- `workspace.PRXBI_DW.wc_event_ed_d` - Event Edition dimension
- `workspace.PRXBI_DW.wc_badge_details_d` - Badge Details dimension
- `workspace.PRXBI_DW.wc_badge_product_d` - Badge Product dimension
- `workspace.PRXBI_DW.wc_event_d` - Event dimension

In [ ]:
%sql
-- Remove widgets at the end of the job (optional)
-- REMOVE WIDGET ETL_JOB_TYPE;
-- REMOVE WIDGET DATASOURCE_NUM_ID;
-- REMOVE WIDGET ETL_PROC_WID;